# Computer Vision 101 — 90 นาที

โน้ตบุ๊กนี้ **กด Run all แล้วรอ** ไม่มีเซลล์ไหนต้องแก้ค่าก่อนรัน

สารของวันนี้: **การทำให้ "พอใช้ได้" นั้นง่าย แต่การทำให้ "ใช้งานได้จริง" นั้นยาก**

- พาร์ท 1 — สอนโมเดลจิ๋วให้เจอ "แก้ว" ด้วยรูป 10 ใบ
- พาร์ท 2 — จับ 21 จุดบนมือ แล้วเขียนกฎ กำ/แบ เอง
- พาร์ท 3 — เอาสองอย่างมาต่อกันเป็น "คนถือแก้ว" ด้วยกฎ 3 บรรทัด

In [ ]:
# เซลล์ 0 — ติดตั้ง (pin ไว้ไม่ให้โน้ตบุ๊กเน่าเมื่อ Colab อัปเดต)
!pip install -q ultralytics==8.3.* mediapipe==1.0.1

In [ ]:
# เซลล์ 1 — preflight: ถ้าอะไรผิด ให้พังตรงนี้ ดีกว่าไปพังนาทีที่ 60
import sys, torch, ultralytics, mediapipe, cv2
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| GPU:", torch.cuda.is_available())
print("ultralytics:", ultralytics.__version__)
print("mediapipe  :", mediapipe.__version__)
assert ultralytics.__version__.startswith("8.3"), "ultralytics เวอร์ชันไม่ตรง"
print("\nพร้อมแล้ว — ไม่ต้องมี GPU ก็รันได้ทั้งโน้ตบุ๊ก")

In [ ]:
# เซลล์ 2 — helper กล้อง: หัวใจของทั้งไฟล์ เขียนครั้งเดียว ใช้ 3 ที่
# เปิดกล้องผ่าน browser (Colab รัน cv2.VideoCapture(0) ไม่ได้) ส่งเฟรม BGR ให้ process_frame
import time, io
from base64 import b64decode, b64encode
import numpy as np, cv2, PIL.Image
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js

_WEBCAM_JS = Javascript("""
var _v, _stream, _canvas, _ctx, _out, _stopped;
async function webcamStart() {
  _stopped = false;
  _v = document.createElement('video'); _v.style.display = 'none';
  _stream = await navigator.mediaDevices.getUserMedia({video: true});
  _v.srcObject = _stream; await _v.play();
  _canvas = document.createElement('canvas');
  _canvas.width = _v.videoWidth; _canvas.height = _v.videoHeight;
  _ctx = _canvas.getContext('2d');
  var btn = document.createElement('button');
  btn.textContent = 'Stop'; btn.style.margin = '8px';
  btn.onclick = () => { _stopped = true; };
  _out = document.createElement('img');
  var div = document.createElement('div');
  div.appendChild(btn); div.appendChild(document.createElement('br')); div.appendChild(_out);
  document.body.appendChild(div);
}
function webcamFrame() {
  if (_stopped) return '';
  _ctx.drawImage(_v, 0, 0);
  return _canvas.toDataURL('image/jpeg', 0.8);
}
function webcamShow(d) { if (_out) _out.src = d; }
function webcamStop() {
  _stopped = true;
  if (_stream) _stream.getTracks().forEach(t => t.stop());
}
""")

def run_webcam(process_frame, seconds=20):
    """เปิดกล้อง browser เรียก process_frame(bgr)->bgr ทุกเฟรม กด Stop หรือครบ seconds เพื่อจบ"""
    display(_WEBCAM_JS)
    try:
        eval_js("webcamStart()")
    except Exception:
        print("เปิดกล้องไม่ได้ — เบราว์เซอร์บล็อกกล้อง หรือใช้ Safari (getUserMedia งอแงที่สุด)\n"
              "ลอง: กด Allow ตอน popup ขึ้น / เปลี่ยนไปใช้ Chrome / รันเซลล์นี้ใหม่")
        return
    end = time.time() + seconds
    while time.time() < end:
        data = eval_js("webcamFrame()")
        if not data:
            break
        _, b64 = data.split(",", 1)
        bgr = cv2.imdecode(np.frombuffer(b64decode(b64), np.uint8), cv2.IMREAD_COLOR)
        out = process_frame(bgr)
        _, buf = cv2.imencode(".jpg", out)
        eval_js('webcamShow("data:image/jpeg;base64,%s")' % b64encode(buf).decode())
    eval_js("webcamStop()")

def run_video(path, process_frame, seconds=20):
    """เล่นไฟล์วิดีโอแทนกล้อง — เผื่อกล้องพังหน้างาน ให้อัดคลิปสั้นๆ แล้วชี้ path มาที่ไฟล์"""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    for _ in range(int(fps * seconds)):
        ok, frame = cap.read()
        if not ok:
            break
        _, buf = cv2.imencode(".jpg", process_frame(frame))
        clear_output(wait=True)
        display(PIL.Image.open(io.BytesIO(buf)))
    cap.release()

### ถ้ากล้องใช้ไม่ได้
- popup ขออนุญาตกล้องขึ้นมาให้กด **Allow** — ถ้าเผลอกดปฏิเสธ ต้องกดที่ไอคอนกล้องบน address bar แล้วอนุญาตใหม่
- **Safari งอแงที่สุด** ให้เปลี่ยนไปใช้ Chrome
- ยังไม่ได้จริงๆ: อัดคลิปสั้นๆ ด้วยมือถือ อัปโหลดเข้า Colab แล้วใช้ `run_video("clip.mp4", process_frame)` แทน `run_webcam(...)`

---
## พาร์ท 1 — Object Detection

โมเดลจะตอบกลับมาเป็น **กล่อง (bounding box)** รอบวัตถุ พร้อม

- **label** — ชื่อคลาส (ของเรามีคลาสเดียว: `cup`)
- **conf** — ความมั่นใจ 0-1 ยิ่งสูงยิ่งมั่นใจ เราจะตัดที่ 0.25

โมเดลไม่รู้จักคำว่า "แก้ว" มาก่อน มันเรียนจากตัวอย่างที่เรา *ชี้ให้ดู* ว่ากล่องนี้คือแก้ว
นั่นคือหน้าที่ของ label

In [ ]:
# เซลล์ 4 — โหลด data (clone ตรงๆ ไม่พึ่ง submodule เพื่อไม่ให้ได้โฟลเดอร์ว่างเงียบๆ)
!git clone -q https://github.com/P-PrPas/tkk_workshop-data.git data
!ls data

In [ ]:
# เซลล์ 5 — ตรวจฟอร์แมต label ก่อน แล้วค่อยดูด้วยตา
# label ผิดฟอร์แมตจะเทรนผ่านโดยไม่มี error แต่ได้โมเดลที่ตรวจไม่เจออะไรเลย
from pathlib import Path
import matplotlib.pyplot as plt

splits = ["train", "val", "test"]
imgs = {s: sorted(Path(f"data/images/{s}").glob("*.jpg")) for s in splits}
image_names = {p.name for s in splits for p in imgs[s]}

for s in splits:
    for txt in Path(f"data/labels/{s}").glob("*.txt"):
        assert txt.with_suffix(".jpg").name in image_names, f"{txt.name} ไม่มีรูปคู่กัน"
        for line in txt.read_text().splitlines():
            if not line.strip():
                continue
            cls, *box = line.split()
            assert cls == "0", f"{txt.name}: class ต้องเป็น 0"
            assert all(0 <= float(v) <= 1 for v in box), f"{txt.name}: พิกัดต้อง normalize 0-1"
print("label ผ่านการตรวจทั้งหมด")

# วาดกล่อง ground-truth จากไฟล์ .txt — เห็นกล่องอยู่ถูกที่ = การตรวจที่ครอบคลุมกว่า assert
all_imgs = [p for s in splits for p in imgs[s]]
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for ax, p in zip(axes.ravel(), all_imgs):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    lbl = Path(str(p).replace("/images/", "/labels/")).with_suffix(".txt")
    for line in lbl.read_text().splitlines() if lbl.exists() else []:
        if not line.strip():
            continue
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax.imshow(im); ax.axis("off"); ax.set_title(p.name, fontsize=8)
for ax in axes.ravel()[len(all_imgs):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

print("train:", len(imgs["train"]), "| val:", len(imgs["val"]), "| test:", len(imgs["test"]))
# 10 / 2 / 3  — ใช่ครับ สิบรูป จำตัวเลขนี้ไว้

### 1.2 — เทรน

เริ่มจาก `yolo11n.pt` (ไม่ใช่ `.yaml`) คือน้ำหนักที่ผ่าน COCO มาแล้ว
`seed=0` ให้ทุกคนได้ผลใกล้กัน · `batch=4` เพราะ train มีแค่ 10 รูป · 3 epoch จบใน ~2 นาทีบน CPU

In [ ]:
# เซลล์ 6 — เทรน
from ultralytics import YOLO
model = YOLO("yolo11n.pt")          # เริ่มจากน้ำหนักที่ผ่าน COCO มาแล้ว
model.train(data="data/cup.yaml", epochs=3, imgsz=640, batch=4, seed=0, plots=True)

### จุดที่ห้ามข้าม

เราไม่ได้เทรนโมเดลนี้ขึ้นมาจากศูนย์ เราเริ่มจาก `yolo11n.pt` ที่เห็นรูปมาแล้วแสนกว่ารูป
จาก COCO ซึ่ง**มีคลาส "cup" อยู่แล้ว** สิ่งที่รูป 10 ใบของเราทำคือ *ขยับ* โมเดลให้เข้ากับ
แก้วในห้องนี้เท่านั้น

ถ้าเริ่มจากศูนย์จริงๆ ด้วยข้อมูลเท่านี้ มันจะตรวจไม่เจออะไรเลย — นี่คือบทเรียน transfer learning
ที่จริงกว่าการแกล้งทำเป็นว่าเราเทรนสำเร็จเอง

In [ ]:
# เซลล์ 8 — หลักฐานหลัก: ดูผลบน test set ด้วยตา (ต้องมีอย่างน้อย 1 รูปที่โมเดลพลาด)
import matplotlib.pyplot as plt
test_imgs = sorted(Path("data/images/test").glob("*.jpg"))
fig, axes = plt.subplots(1, len(test_imgs), figsize=(6 * len(test_imgs), 6))
for ax, p in zip(np.atleast_1d(axes), test_imgs):
    r = model(str(p), conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title(p.name)
plt.tight_layout(); plt.show()

In [ ]:
# เซลล์ 9 — ของแถม: ตัวเลข mAP
metrics = model.val(split="test")
print("mAP50:", round(metrics.box.map50, 3))

val มี 2 รูป — ตัวเลขนี้ขยับทีละ ~50% ต่อหนึ่งรูป **อย่าเอาไปอ้างอิงที่ไหน**
ดูกริดรูปข้างบนเถอะ นั่นคือหลักฐานที่เชื่อได้กว่า

### 1.4 — Realtime ผ่านกล้อง

ลองเอาแก้วเข้า-ออกเฟรม เอียงแก้ว เอามือบัง แล้วลองเอา **ขวดน้ำ** มาให้ดู
(มันจะทายว่าเป็นแก้ว หรือไม่เห็นเลย — ทั้งสองอย่างคือบทเรียน)

In [ ]:
# เซลล์ 10 — realtime detection
def process_frame(bgr):
    r = model(bgr, conf=0.25, verbose=False)[0]
    return r.plot()

run_webcam(process_frame, seconds=20)
# run_video("clip.mp4", process_frame)   # ← กล้องพัง? อัปโหลดคลิป แล้วลบ # บรรทัดนี้แทน run_webcam

---
## พาร์ท 2 — Hand Pose

Detection ตอบว่า *"มีแก้วอยู่ตรงนี้"* — Pose ตอบละเอียดกว่านั้นว่า *"ข้อนิ้วแต่ละข้ออยู่พิกัดไหน"*
MediaPipe ให้ **21 จุด** ต่อมือหนึ่งข้าง เราจะเอา 21 จุดนั้นมาเขียนกฎเองว่ามือ *กำ* หรือ *แบ*

In [ ]:
# เซลล์ 12 — โหลดโมเดล hand landmarker (float16)
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!ls -la hand_landmarker.task

In [ ]:
# เซลล์ 13 — กติกา กำ/แบ เขียนเอง
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

_opts = mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=mp_vision.RunningMode.VIDEO, num_hands=2)
landmarker = mp_vision.HandLandmarker.create_from_options(_opts)

TIPS = [4, 8, 12, 16, 20]
PIPS = [2, 6, 10, 14, 18]

def count_extended(lm):
    """นิ้วเหยียด = ปลายนิ้วอยู่ไกลจากข้อมือกว่าข้อกลาง"""
    w = lm[0]
    d = lambda p: (p.x - w.x) ** 2 + (p.y - w.y) ** 2
    return sum(d(lm[t]) > d(lm[p]) for t, p in zip(TIPS, PIPS))

def hand_state(lm):
    n = count_extended(lm)
    return "FIST" if n <= 1 else "OPEN" if n >= 4 else "UNKNOWN"

ทำไมไม่ใช้ `tip.y < pip.y` (ปลายนิ้วอยู่สูงกว่าข้อ)? **เพราะพอเอียงมือหรือชี้ลง มันพังทันที**
เราวัด*ระยะจากข้อมือ*แทน ซึ่งทนการหมุนได้

In [ ]:
# เซลล์ 14 — realtime hand
import mediapipe as mp
_t = [0]

def hand_process_frame(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    _t[0] += 33
    res = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), _t[0])
    h, w = bgr.shape[:2]
    for lm in (res.hand_landmarks or []):
        pts = [(int(p.x * w), int(p.y * h)) for p in lm]
        for x, y in pts:
            cv2.circle(bgr, (x, y), 4, (0, 255, 0), -1)
        cv2.putText(bgr, hand_state(lm), pts[0], cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)
    return bgr

run_webcam(hand_process_frame, seconds=20)
# run_video("clip.mp4", hand_process_frame)   # ← กล้องพัง? อัปโหลดคลิป แล้วลบ # บรรทัดนี้

ลองกำๆ แบๆ ค้างระหว่างกลาง — ป้ายจะ **กระพริบ** ระหว่าง FIST / UNKNOWN / OPEN
จำอาการนี้ไว้ เดี๋ยวเรากลับมาแก้มันในแอป

---
## พาร์ท 3 — คนถือแก้ว

เราไม่ได้เทรนโมเดล "คนถือแก้ว" — เราเอาผลของสองโมเดลมาต่อกันด้วยกฎ:

> **มือกำ** และ **กล่องมือซ้อนกับกล่องแก้ว** → กำลังถือ

In [ ]:
# เซลล์ 16 — กฎ 3 บรรทัด
def boxes_overlap(a, b):
    return a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]

def hand_bbox(lm, w, h):
    xs = [p.x * w for p in lm]; ys = [p.y * h for p in lm]
    return [min(xs), min(ys), max(xs), max(ys)]

def is_holding(hand_bbox, hand_st, cup_boxes):
    return hand_st == "FIST" and any(boxes_overlap(hand_bbox, c) for c in cup_boxes)

In [ ]:
# เซลล์ 17 — realtime รวมร่าง
import mediapipe as mp
_tt = [0]

def combined_process_frame(bgr):
    h, w = bgr.shape[:2]
    r = model(bgr, conf=0.25, verbose=False)[0]
    cup_boxes = r.boxes.xyxy.tolist() if r.boxes is not None else []
    for x1, y1, x2, y2 in cup_boxes:
        cv2.rectangle(bgr, (int(x1), int(y1)), (int(x2), int(y2)), (255, 180, 0), 2)

    _tt[0] += 33
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), _tt[0])
    holding = False
    for lm in (res.hand_landmarks or []):
        pts = [(int(p.x * w), int(p.y * h)) for p in lm]
        for x, y in pts:
            cv2.circle(bgr, (x, y), 3, (0, 255, 0), -1)
        if is_holding(hand_bbox(lm, w, h), hand_state(lm), cup_boxes):
            holding = True
    label = "HOLDING" if holding else "NOT HOLDING"
    color = (0, 200, 0) if holding else (0, 0, 255)
    cv2.putText(bgr, label, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.6, color, 4)
    return bgr

run_webcam(combined_process_frame, seconds=20)
# run_video("clip.mp4", combined_process_frame)   # ← กล้องพัง? อัปโหลดคลิป แล้วลบ # บรรทัดนี้

---
## ส่งไม้ต่อให้แอป

ดูด้วยตาตัวเองก่อนว่ามีปัญหาอะไรบ้าง แล้วค่อยอ่านเฉลย:

1. ป้ายกระพริบตลอด — ไม่มีความจำข้ามเฟรม
2. แก้วหายไปเฟรมเดียวแล้วกลับมา = กลายเป็นแก้วใบใหม่
3. ช้า
4. ถือแก้วแบบแบมือ (ประคอง) → ตรวจไม่เจอ เพราะกฎเราบังคับว่าต้องกำ
5. ถอดปลั๊กกล้องแล้วทุกอย่างพัง

**ห้าข้อนี้แหละ คือระยะห่างระหว่าง demo กับ product** — เดี๋ยวผมโชว์ตัวที่แก้ครบแล้ว